# Candidate generation and LightGBM re-ranking

The ranker is trained on S16 (history weeks 11 to 15, labels from weeks 16 to 20) with early stopping on 20% of its users, compared with the single methods on S21 and refitted on S16 and S21 together for the test window.

In [1]:
# Environment, data, snapshots and the collaborative models of every snapshot
import json
import os
import platform
import sys
import time

import lightgbm as lgb
import numpy as np
import pandas as pd

ROOT = os.path.abspath('..')
sys.path.insert(0, os.path.join(ROOT, 'src'))
from sweettv import baselines, cf, data, features, ranker, recs, snapshot  # noqa: E402

RAW, CACHE = os.path.join(ROOT, 'data', 'raw'), os.path.join(ROOT, 'data', 'cache')
SUB_DIR = os.path.join(ROOT, 'submissions')
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 40)
print(platform.platform(), '| python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__,
      '| lightgbm', lgb.__version__)
T0 = time.time()
views = data.load_views(RAW, CACHE)
sched = data.load_schedule(RAW, CACHE)
test_users = pd.Index(data.load_submission(RAW).user_id)
SETTINGS = json.load(open(os.path.join(ROOT, 'modelling', 'cf_results.json')))['settings']
print('collaborative settings chosen in cf_models:', SETTINGS)
lab = sched[sched.labelled]
prog = lab.groupby('tv_show_id').agg(title=('tv_show_title', 'first'), category=('tv_show_category', 'first'),
                                     genre_1=('tv_show_genre_1', 'first'), genre_2=('tv_show_genre_2', 'first'),
                                     genre_3=('tv_show_genre_3', 'first'), year=('year_of_production', 'first'),
                                     director=('director', 'first'), actors=('actors', 'first'))
snaps = {name: snapshot.build(views, sched, cut, end, with_target=wt)
         for name, (cut, end, wt) in {'S16': (16, 20, True), 'S21': (21, 30, True), 'S31': (31, 42, False)}.items()}
fallbacks = {name: baselines.popularity_ranking(s) for name, s in snaps.items()}


def cf_models(name):
    m = cf.Interactions.from_snapshot(snaps[name])
    als_kw = dict(SETTINGS['als'])
    als_kw['factors'] = int(als_kw['factors'])
    return {'als': cf.als(m, **als_kw), 'ppr': cf.ppr(m, **SETTINGS['ppr']), 'rules': cf.association_rules(m, **SETTINGS['rules']),
            'content': cf.content(m, prog, sorted(snaps[name].pool)), 'habit': features.habit_scores(snaps[name], views)}


users = {'S16': snaps['S16'].target.index, 'S21': snaps['S21'].target.index, 'S31': test_users}
tables = {}
for name in ['S16', 'S21', 'S31']:
    t = time.time()
    cfs = cf_models(name)
    tables[name] = features.build(snaps[name], views, users[name], cfs, fallbacks[name], prog)
    print(f'{name}: {len(tables[name]):,} candidate rows for {tables[name].user_id.nunique()} users, {time.time() - t:.0f}s')
FEATS = features.feature_columns(tables['S16'])
print(len(FEATS), 'features:', FEATS)

Windows-11-10.0.26200-SP0 | python 3.12.14 | numpy 2.5.4 | pandas 3.0.6 | lightgbm 4.7.0


collaborative settings chosen in cf_models: {'als': {'factors': 32, 'alpha': 3.0, 'reg': 100.0}, 'ppr': {'alpha': 0.5, 'weighting': 'count'}, 'rules': {'min_support': 30}, 'content': {}}


S16: 505,705 candidate rows for 3787 users, 21s


S21: 840,854 candidate rows for 4030 users, 33s


S31: 444,429 candidate rows for 1260 users, 49s
63 features: ['src_touched', 'src_pop', 'src_als', 'src_ppr', 'src_rules', 'src_content', 'src_habit', 'n_touch', 'sec_sum', 'frac_mean', 'frac_max', 'n_q', 'n_half', 'n_qw', 'q_last1', 'q_last2', 'q_last4', 'n_channels_watched', 'weeks_since_last', 'weeks_since_first', 'weeks_since_last_q', 'hs', 'hs_p0', 'hs_p3', 'exposure', 'touch_rate', 'q_rate', 'win_airings', 'win_channels', 'win_slot_min', 'hist_airings', 'airing_ratio', 'win_airings_user_channels', 'habit_exposure', 'habit_exposure_hist', 'q_per_habit', 'pop_top5', 'pop_q_users', 'pop_q_per_user', 'pop_q_last4', 'category', 'year', 'new_programme', 'u_active_days', 'u_lab_hours', 'u_q_programmes', 'u_q_total', 'u_q_5th', 'u_active_last_week', 'cf_als', 'cf_ppr', 'cf_rules', 'cf_content', 'cf_habit', 'rank_hs', 'rank_hs_p0', 'rank_n_qw', 'rank_habit_exposure', 'rank_cf_als', 'rank_cf_ppr', 'rank_cf_rules', 'rank_cf_content', 'rank_cf_habit']


In [2]:
# Candidate recall: share of target pairs that reach the ranker, overall and by source
rows = {}
for name in ['S16', 'S21']:
    df, s = tables[name], snaps[name]
    tp = s.target.explode().astype('int64').reset_index()
    m = tp.merge(df, on=['user_id', 'tv_show_id'], how='left')
    src = [c for c in df.columns if c.startswith('src_')]
    rows[name] = {'candidates per user, median': df.groupby('user_id').size().median(), 'recall of all sources': m.src_pop.notna().mean(),
                  **{f'recall {c[4:]}': (m[c] == 1).mean() for c in src}}
display(pd.DataFrame(rows).round(4))

,S16,S21
"candidates per user, median",128.0000,194.0000
recall of all sources,0.7538,0.7571
recall touched,0.6341,0.6536
recall pop,0.6226,0.6220
recall als,0.5529,0.5580
recall ppr,0.5977,0.5771
recall rules,0.5730,0.5711
recall content,0.3647,0.3320
recall habit,0.6382,0.6223


In [3]:
# Train each objective on S16 and evaluate on S21
def map_s21(score, df=None, positive_only=False):
    df = tables['S21'] if df is None else df
    rank_score = ranker.user_rank_score(df, score)
    if positive_only:
        # a single heuristic leaves its empty places to the popularity fallback, as in the baselines notebook
        rank_score = np.where(np.asarray(score) > 0, rank_score, 0.0)
    sc = df[['user_id', 'tv_show_id']].assign(score=rank_score)
    r = recs.top5(sc, snaps['S21'].target.index, fallbacks['S21'], snaps['S21'].pool)
    return recs.evaluate(r, snaps['S21'].target, {'test users': test_users})


results, preds, best_iters, boosters = {}, {}, {}, {}
for obj in ranker.OBJECTIVES:
    t = time.time()
    b, it = ranker.fit(tables['S16'], obj)
    boosters[obj], best_iters[obj] = b, it
    preds[obj] = ranker.predict(b, tables['S21'])
    results[obj] = {**map_s21(preds[obj]), 'best iteration': it, 'seconds': round(time.time() - t)}
results['habit x schedule feature alone'] = map_s21(tables['S21'].hs.fillna(0).to_numpy(), positive_only=True)
display(pd.DataFrame(results).T.round(4))

,all,test users,best iteration,seconds
binary,0.3539,0.3915,138.0,46.0
lambdarank,0.3566,0.3956,98.0,44.0
poisson,0.3510,0.3877,554.0,117.0
habit x schedule feature alone,0.3366,0.3770,NaN,NaN


In [4]:
# Ensembles: averaged reciprocal ranks of the objectives
rr = {obj: ranker.user_rank_score(tables['S21'], p) for obj, p in preds.items()}
ens = {
    'binary + lambdarank': rr['binary'] + rr['lambdarank'],
    'binary + poisson': rr['binary'] + rr['poisson'],
    'lambdarank + poisson': rr['lambdarank'] + rr['poisson'],
    'all three': rr['binary'] + rr['lambdarank'] + rr['poisson'],
}
ens_res = {k: map_s21(v) for k, v in ens.items()}
display(pd.DataFrame(ens_res).T.round(4))

,all,test users
binary + lambdarank,0.3554,0.3935
binary + poisson,0.3537,0.3908
lambdarank + poisson,0.3548,0.3917
all three,0.3568,0.3947


In [5]:
# Feature importance (gain) of the binary model
imp = pd.Series(boosters['binary'].feature_importance('gain'), index=boosters['binary'].feature_name()).sort_values(ascending=False)
display((imp / imp.sum()).head(30).round(4).to_frame('gain share').T)

,rank_habit_exposure,hs,rank_cf_habit,n_half,habit_exposure,pop_q_per_user,rank_cf_rules,cf_rules,u_q_total,airing_ratio,win_airings,u_lab_hours,win_airings_user_channels,sec_sum,cf_content,rank_cf_content,rank_cf_ppr,pop_top5,weeks_since_last,frac_mean,n_touch,rank_hs,touch_rate,cf_ppr,win_slot_min,cf_als,rank_hs_p0,weeks_since_first,hist_airings,u_q_programmes
gain share,0.3166,0.2145,0.1206,0.0382,0.0276,0.0257,0.0213,0.0185,0.0129,0.0114,0.0103,0.0103,0.0097,0.0087,0.0086,0.0085,0.0084,0.0066,0.0063,0.0061,0.0061,0.006,0.0057,0.0055,0.0053,0.0052,0.0049,0.0049,0.0048,0.0044


In [6]:
# Final models: refit every objective on S16 and S21 with 1.2 times the best iteration, predict the test window
final_train = pd.concat([tables['S16'], tables['S21']], ignore_index=True)
t31 = tables['S31']
p31 = {}
for obj in ranker.OBJECTIVES:
    b = ranker.refit(final_train, obj, int(best_iters[obj] * 1.2))
    p31[obj] = ranker.predict(b, t31)
s = snaps['S31']
out = {}
for name, score in {'ranker_binary': p31['binary'], 'ranker_lambdarank': ranker.user_rank_score(t31, p31['lambdarank']),
                    'ranker_poisson': p31['poisson'],
                    'ranker_ensemble': sum(ranker.user_rank_score(t31, p31[o]) for o in ranker.OBJECTIVES)}.items():
    sc = t31[['user_id', 'tv_show_id']].assign(score=ranker.user_rank_score(t31, score))
    r = recs.top5(sc, test_users, fallbacks['S31'], s.pool)
    recs.write_submission(r, test_users, os.path.join(SUB_DIR, f'{name}.csv'))
    out[name] = r
print(sorted(os.listdir(SUB_DIR)))
with open(os.path.join(ROOT, 'modelling', 'ranker_results.json'), 'w') as f:
    json.dump({'results': results, 'ensembles': ens_res, 'best_iterations': best_iters,
               'importance': (imp / imp.sum()).round(5).to_dict()}, f, indent=1, default=str)
print(f'total runtime {time.time() - T0:.0f}s')

['als.csv', 'content.csv', 'habit_schedule.csv', 'popularity.csv', 'ppr.csv', 'ranker_binary.csv', 'ranker_ensemble.csv', 'ranker_lambdarank.csv', 'ranker_poisson.csv', 'rules.csv']
total runtime 497s
